# Track A: the news MCP wrapper on Container Apps

Put a server you own in front of the GNews MCP server, give `news-desk-agent` a tool that reaches it with the user's delegated token, and show that user A and user B get different news from the same agent.

[`news-mcp-wrapper/server.py`](news-mcp-wrapper/server.py) is a FastMCP server. It validates the Entra token Foundry sends on the user's behalf, lists only the tools the token's app roles grant, answers any other call with a denial that names the allowed categories, and forwards allowed calls to GNews with the GNews key, which never leaves the container. [08-05c-00](08-05c-00-news-mcp-per-user-access.md) compares it with the gateway approach of [08-05c-03](08-05c-03-apim-passthrough.ipynb).

**What this notebook does:**
- **Step 0** sets up names, credentials and helpers
- **Step 1** creates `rg-foundry-news-mcp-{suffix}` and deploys the registry, the pull identity, the log workspace and the Container Apps environment
- **Step 2** builds the wrapper image in the registry
- **Step 3** deploys the container app and waits for `/health`
- **Step 4** creates the `news-mcp-wrapper` connection and a `news-desk-agent` version that uses it
- **Sign-in** signs in user A and user B
- **Step 5** reads the wrapper's log to see which claims the delegated token carries
- **Step 6** runs the test matrix as both users
- **Step 7** compares each user's `tools/list`, through the agent and directly
- **Step 8** revokes user B's `News.Read.Science`, which 08-05c-03's science case then expects to be denied

## Prerequisites

1. **08-05c-01** has run: it wrote the app, user, blueprint and connection settings to `.env`.
2. **A GNews key**: register at [gnews.io](https://gnews.io) (free plan, 100 requests a day) and add `GNEWS_API_KEY=...` to `.env`. This notebook makes about four GNews requests; denied calls make none.
3. **Azure CLI** as in 08-05c-01. You need Owner (or Contributor plus User Access Administrator) on the subscription, to create the resource group and the AcrPull role assignment.
4. **User B's pass** from 08-05c-01 is valid for 8 hours. If it has expired, re-run Step 5 of 08-05c-01 for a new one.
5. Optional: `NEWS_MCP_RESOURCE_GROUP` and `NEWS_MCP_LOCATION` in `.env` override the resource group name and region (default `swedencentral`).

## Step 0: Set up names, credentials and helpers

Loads `.env`, derives the lab's names from your Azure CLI sign-in (subscription, tenant, suffix, the Foundry project, the resource group and the container app), and defines the helpers the later steps use to call Microsoft Graph, ARM, the Azure CLI and Foundry. It only reads: nothing is created yet. The cell stops if `.env` lacks `GNEWS_API_KEY` or a value that 08-05c-01 writes.

In [1]:
import hashlib, json, os, subprocess, tempfile, time
from datetime import datetime, timezone
from pathlib import Path

import requests
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

import news_mcp_checks as checks
import news_mcp_identity as ident
from news_mcp_identity import authz

repo_root = Path(subprocess.run('git rev-parse --show-toplevel', shell=True, capture_output=True, text=True).stdout.strip())
ENV_FILE = repo_root / '.env'
load_dotenv(ENV_FILE, override=True)

REQUIRED = ['NEWS_MCP_RESOURCE_APP_ID', 'NEWS_MCP_RESOURCE_SP_ID', 'NEWS_DESK_CLIENT_APP_ID', 'NEWS_TEST_USER_UPN',
            'NEWS_TEST_USER_ID', 'NEWS_AGENT_BLUEPRINT_CLIENT_ID', 'NEWS_CONNECTION_AUTH_TYPE',
            'NEWS_CONNECTION_API_VERSION', 'GNEWS_API_KEY']
missing = [key for key in REQUIRED if not os.environ.get(key)]
if missing:
    raise SystemExit(f"Missing in .env: {', '.join(missing)}. Run 08-05c-01 first, and add GNEWS_API_KEY.")

account = json.loads(subprocess.run('az account show -o json', shell=True, capture_output=True, text=True).stdout)
SUBSCRIPTION_ID, TENANT_ID = account['id'], account['tenantId']
SUFFIX = hashlib.sha256((SUBSCRIPTION_ID + 'v2').encode()).hexdigest()[:6]
CHAT_MODEL = os.environ.get('CHAT_MODEL', 'gpt-5.4-mini')
PROJECT_ENDPOINT = f'https://aif-core-{SUFFIX}.services.ai.azure.com/api/projects/project-admin-{SUFFIX}'

RESOURCE_APP_ID = os.environ['NEWS_MCP_RESOURCE_APP_ID']
RESOURCE_SP_ID = os.environ['NEWS_MCP_RESOURCE_SP_ID']
CLIENT_APP_ID = os.environ['NEWS_DESK_CLIENT_APP_ID']
BLUEPRINT_CLIENT_ID = os.environ['NEWS_AGENT_BLUEPRINT_CLIENT_ID']
TEST_USER_ID = os.environ['NEWS_TEST_USER_ID']
AUTH_TYPE, CONNECTION_API = os.environ['NEWS_CONNECTION_AUTH_TYPE'], os.environ['NEWS_CONNECTION_API_VERSION']

RG = os.environ.get('NEWS_MCP_RESOURCE_GROUP') or f'rg-foundry-news-mcp-{SUFFIX}'
LOCATION = os.environ.get('NEWS_MCP_LOCATION') or 'swedencentral'
APP_NAME = f'ca-news-mcp-{SUFFIX}'
LAB_DIR = Path.cwd()

cli = AzureCliCredential()
ARM, FOUNDRY = 'https://management.azure.com/.default', 'https://ai.azure.com/.default'
project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=cli, allow_preview=True)


def graph(method, path, body=None):
    """Microsoft Graph v1.0 as the Azure CLI user."""
    r = requests.request(method, f'https://graph.microsoft.com/v1.0{path}', json=body, timeout=60,
                         headers={'Authorization': f"Bearer {cli.get_token('https://graph.microsoft.com/.default').token}"})
    if not r.ok:
        raise RuntimeError(f'{method} {path}: HTTP {r.status_code} {r.text[:600]}')
    return r.json() if r.content else None


def call(method, url, scope, body=None):
    """ARM or the Foundry data plane with a CLI token for `scope`. The caller checks the response."""
    return requests.request(method, url, data=None if body is None else json.dumps(body), timeout=120,
                            headers={'Authorization': f'Bearer {cli.get_token(scope).token}', 'Content-Type': 'application/json'})


def env_set(**values):
    """Write keys to the repo .env: replace the key's line if present, append it otherwise."""
    lines = ENV_FILE.read_text().splitlines()
    for key, value in values.items():
        os.environ[key] = value
        at = [i for i, line in enumerate(lines) if line.split('=', 1)[0].strip() == key]
        if at:
            lines[at[0]] = f'{key}={value}'
        else:
            lines.append(f'{key}={value}')
    ENV_FILE.write_text('\n'.join(lines) + '\n')


def agent_identity(name):
    """The agent identity (principal_id, client_id) of an agent's latest version."""
    r = call('GET', f'{PROJECT_ENDPOINT}/agents/{name}?api-version={ident.AGENT_API_VERSION}', FOUNDRY)
    r.raise_for_status()
    return r.json()['versions']['latest']['instance_identity']


def az(*args):
    """Run an az command; return its JSON output, or raise with the end of stderr."""
    r = subprocess.run(['az', *args, '-o', 'json'], capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"az {' '.join(args[:3])}: {r.stderr.strip()[-1500:]}")
    return json.loads(r.stdout) if r.stdout.strip() else None


print(f'Suffix          : {SUFFIX}')
print(f'Resource group  : {RG} ({LOCATION})')
print(f'Container app   : {APP_NAME}')
print(f'Connection auth : {AUTH_TYPE} (ARM api-version {CONNECTION_API})')

Suffix          : c2676f
Resource group  : rg-foundry-news-mcp-c2676f (swedencentral)
Container app   : ca-news-mcp-c2676f
Connection auth : AgentOnBehalfOfUser (ARM api-version 2025-04-01-preview)


## Step 1: Create the resource group and what the container app needs

[`infra/wrapper.bicep`](infra/wrapper.bicep) is deployed twice, because the container app needs an image that does not exist yet. This first pass (`deployApp=false`) creates everything except the app:

| Resource | What it is for |
|---|---|
| Log Analytics workspace `log-news-mcp-{suffix}` | receives the container app's logs |
| container registry `acrnewsmcp{suffix}` | Basic tier, no admin user; Step 2 builds the image here |
| user-assigned identity `id-news-mcp-{suffix}` | holds AcrPull on the registry, so the container app pulls the image without a password |
| Container Apps environment `cae-news-mcp-{suffix}` | where Step 3 puts the container app |

The resource group carries the tag `SecurityControl=Ignore`. Some governed subscriptions run a policy that rewrites public network settings, and this tag excludes the group from it. The GNews key reaches the deployment through a parameters file that the cell deletes right after, so the key never appears on a command line.

The cell prints how long the deployment took and the resources now in the group.

In [2]:
az('group', 'create', '-n', RG, '-l', LOCATION, '--tags', 'SecurityControl=Ignore')
AUDIENCES = [RESOURCE_APP_ID, authz.app_id_uri(SUFFIX)]


def deploy(deploy_app, image=''):
    """Deploy infra/wrapper.bicep and return its outputs."""
    params = {'suffix': SUFFIX, 'tenantId': TENANT_ID, 'audiences': AUDIENCES,
              'gnewsApiKey': os.environ['GNEWS_API_KEY'], 'deployApp': deploy_app, 'image': image}
    with tempfile.NamedTemporaryFile('w', suffix='.json', delete=False) as f:
        json.dump({'$schema': 'https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#',
                   'contentVersion': '1.0.0.0', 'parameters': {k: {'value': v} for k, v in params.items()}}, f)
    try:
        result = az('deployment', 'group', 'create', '-g', RG, '-n', 'news-mcp-wrapper',
                    '--template-file', str(LAB_DIR / 'infra' / 'wrapper.bicep'), '--parameters', f'@{f.name}')
    finally:
        os.unlink(f.name)
    return {k: v['value'] for k, v in result['properties']['outputs'].items()}


t0 = time.time()
outputs = deploy(False)
REGISTRY, LOGIN_SERVER = outputs['registryName'], outputs['registryLoginServer']
print(f'Deployed in {time.time() - t0:.0f}s')
for resource in az('resource', 'list', '-g', RG, '--query', '[].{type:type, name:name}'):
    print(f"  {resource['type']:<55} {resource['name']}")

Deployed in 40s
  Microsoft.ManagedIdentity/userAssignedIdentities        id-news-mcp-c2676f
  Microsoft.ContainerRegistry/registries                  acrnewsmcpc2676f
  Microsoft.OperationalInsights/workspaces                log-news-mcp-c2676f
  Microsoft.App/managedEnvironments                       cae-news-mcp-c2676f
  Microsoft.App/containerApps                             ca-news-mcp-c2676f


## Step 2: Build the wrapper image in the registry

`az acr build` sends [`news-mcp-wrapper/`](news-mcp-wrapper/) to the registry, which builds the image there, so this machine does not need Docker. The cell prints the image reference with its digest. The container app refers to the image by that digest, so every rebuild becomes a new revision.

In [3]:
t0 = time.time()
r = subprocess.run(['az', 'acr', 'build', '-r', REGISTRY, '-t', 'news-mcp-wrapper:latest', '--platform', 'linux/amd64',
                    '--no-logs', str(LAB_DIR / 'news-mcp-wrapper')], capture_output=True, text=True)
if r.returncode:
    raise RuntimeError(r.stderr[-1500:])
digest = az('acr', 'repository', 'show', '-n', REGISTRY, '--image', 'news-mcp-wrapper:latest', '--query', 'digest')
IMAGE = f'{LOGIN_SERVER}/news-mcp-wrapper@{digest}'
print(f'Built in {time.time() - t0:.0f}s: {IMAGE}')

Built in 68s: acrnewsmcpc2676f.azurecr.io/news-mcp-wrapper@sha256:c0f599a3f7c1130c6cd2d46e3f08645f8d3ea9079ce28b3572ac9464cdf8db97


## Step 3: Deploy the container app

The second pass of the template (`deployApp=true`) adds the container app `ca-news-mcp-{suffix}`:

| Setting | Value |
|---|---|
| Ingress | external HTTPS, to port 8080 |
| Replicas | one to two. One is always on, because Foundry gives an MCP call 100 seconds and a cold start would use part of that |
| GNews key | a Container Apps secret, passed to the container as `GNEWS_API_KEY` |
| Probes | liveness and readiness on `/health` |

The cell waits until `/health` answers, then sends two requests without a token: `GET /health`, which is open and returns 200, and `POST /mcp`, which the wrapper refuses with 401. It writes the wrapper's URL to `.env` as `NEWS_MCP_WRAPPER_URL`.

In [4]:
t0 = time.time()
outputs = deploy(True, IMAGE)
WRAPPER_MCP_URL = outputs['wrapperMcpUrl']
base = WRAPPER_MCP_URL.removesuffix('/mcp')
for _ in range(60):
    try:
        if requests.get(f'{base}/health', timeout=10).status_code == 200:
            break
    except requests.RequestException:
        pass
    time.sleep(5)
print(f'Deployed in {time.time() - t0:.0f}s: {WRAPPER_MCP_URL}')
print(f"GET  /health without a token: HTTP {requests.get(f'{base}/health', timeout=10).status_code}")
print(f"POST /mcp    without a token: HTTP {requests.post(WRAPPER_MCP_URL, json={}, timeout=10).status_code}")
env_set(NEWS_MCP_WRAPPER_URL=WRAPPER_MCP_URL)

Deployed in 70s: https://ca-news-mcp-c2676f.purpletree-ca1d7524.swedencentral.azurecontainerapps.io/mcp
GET  /health without a token: HTTP 200
POST /mcp    without a token: HTTP 401


## Step 4: Give `news-desk-agent` an MCP tool that calls the wrapper

The cell creates two things:

1. **The `news-mcp-wrapper` connection** in the Foundry project. It is the same kind of connection as `news-mcp-apim` from 08-05c-01, with the same auth type and the same `scopes`, but its target is the wrapper's URL. It tells Foundry to get a `News.Read` token on the user's behalf before each call.
2. **A new version of `news-desk-agent`** with one MCP tool, `news_mcp_wrapper`, that names that connection. Agents are versioned: each change to the definition adds a version, and the agent endpoint runs the latest one.

The cell prints the connection's auth type and scopes, then the new agent version and the tool's URL.

In [5]:
def put_connection(name, body):
    r = call('PUT', ident.connection_url(SUBSCRIPTION_ID, SUFFIX, name, CONNECTION_API), ARM, body)
    if not r.ok:
        raise RuntimeError(f'PUT connection {name}: HTTP {r.status_code} {r.text[:600]}')
    return r.json()


connection = put_connection('news-mcp-wrapper', ident.delegated_connection_body(WRAPPER_MCP_URL, SUFFIX, AUTH_TYPE))
print(f"Connection news-mcp-wrapper: authType={connection['properties']['authType']}  scopes={connection['properties'].get('scopes')}")

agent = project.agents.create_version(agent_name=ident.AGENT_NAME, definition=PromptAgentDefinition(
    model=CHAT_MODEL, instructions=ident.AGENT_INSTRUCTIONS,
    tools=[ident.mcp_tool('news_mcp_wrapper', WRAPPER_MCP_URL, connection['id'])]))
TRACK_A_VERSION = agent.version
print(f'{agent.name} v{agent.version}: tool news_mcp_wrapper -> {WRAPPER_MCP_URL}')

Connection news-mcp-wrapper: authType=AgentOnBehalfOfUser  scopes=['api://news-mcp-c2676f/News.Read']
news-desk-agent v9: tool news_mcp_wrapper -> https://ca-news-mcp-c2676f.purpletree-ca1d7524.swedencentral.azurecontainerapps.io/mcp


## Sign in as user A and user B

Two device code sign-ins, as in 08-05c-01. Each cell prints the account it expects and stops if the token belongs to someone else.

| Cell | Sign in as | Account | Method |
|---|---|---|---|
| first | user A | your admin account, the one `az login` uses | your usual password and multifactor method |
| second | user B | `news-sports-desk@{your tenant's default domain}`, in `.env` as `NEWS_TEST_USER_UPN` | Temporary Access Pass, in `.env` as `NEWS_TEST_USER_TAP` |

`grep -E 'NEWS_TEST_USER_(UPN|TAP)' .env` from the repository root shows user B's account and pass.

For user B:

1. Open a private browser window. A normal window offers the account you are already signed in with, and picking it signs in user A instead.
2. Open the address the cell prints and enter its code.
3. The page names `news-desk-client-{suffix}` and the location of the machine that runs the kernel. That is expected: continue.
4. If an account picker appears, choose **Use another account** and type user B's account.
5. If Entra asks for a password, choose **Other ways to sign in**, then **Temporary Access Pass**, and paste the pass.

The second cell then builds one agent client per user from these sign-ins. Each client calls `news-desk-agent` with that user's token for the agent blueprint's `access_agent` scope, and the later steps use them to ask questions as user A or user B.

In [6]:
print('User A, your admin account:')
cred_a = checks.user_credential(TENANT_ID, CLIENT_APP_ID, device_code=True)
token = cred_a.get_token(f'api://{BLUEPRINT_CLIENT_ID}/access_agent').token  # the sign-in; cred_a caches it
if checks.jwt_claims(token)['oid'] == TEST_USER_ID:
    raise RuntimeError('That sign-in was user B. Sign in as user A, your admin account, and re-run this cell.')

User A, your admin account:
To sign in, use a web browser to open the page https://login.microsoft.com/device and enter the code ELQVDHHMR to authenticate.


In [7]:
print(f"User B, {os.environ['NEWS_TEST_USER_UPN']}, in a private window, with the Temporary Access Pass in .env (NEWS_TEST_USER_TAP):")
cred_b = checks.user_credential(TENANT_ID, CLIENT_APP_ID, device_code=True)
token = cred_b.get_token(f'api://{BLUEPRINT_CLIENT_ID}/access_agent').token  # the sign-in
if checks.jwt_claims(token)['oid'] != TEST_USER_ID:
    raise RuntimeError("That sign-in was not user B. A device code page reuses any account already signed in to the browser: open a private window, choose \"Use another account\", sign in as user B with the pass, and re-run this cell.")
print('Signed in as user B')

clients = {user: checks.agent_client(cred, PROJECT_ENDPOINT, ident.AGENT_NAME, BLUEPRINT_CLIENT_ID)
           for user, cred in (('A', cred_a), ('B', cred_b))}
ROLES = {'A': ['News.Read.All'], 'B': ['News.Read.Sports', 'News.Read.Science', 'News.Read.Health']}
print('\nClients ready for user A and user B')

User B, {user}@{tenant}.onmicrosoft.com, in a private window, with the Temporary Access Pass in .env (NEWS_TEST_USER_TAP):
To sign in, use a web browser to open the page https://login.microsoft.com/device and enter the code E44TDXGXD to authenticate.
Signed in as user B

Clients ready for user A and user B


## Step 5: Read the token's claims from the wrapper's log

The cell asks the agent for business headlines as user A, then reads the wrapper's log. The wrapper writes one JSON line per event to stdout, and `az containerapp logs show` returns the recent ones. The cell prints two of them:

- **The `initialize` line**: the name of Foundry's MCP client and the protocol version it asked for.
- **The `tools/call` line**: the claims of the token Foundry obtained on user A's behalf, the tool called and the wrapper's decision.

| Claim | Expected |
|---|---|
| `idtyp` | `user`: a delegated token |
| `oid` | user A's object id |
| `azp` | the client id of the agent identity that asked for the token (not the blueprint) |
| `roles` | `News.Read.All`, from the assignment in 08-05c-01 |
| `scp` | `News.Read` |
| `aud` | the client id of `news-mcp-{suffix}` (v2 token) |

The last line of the output checks that `azp` is the client id of `news-desk-agent`'s agent identity.

In [8]:
def wrapper_events(kind=None, tail=300):
    """The wrapper's recent JSON events from the container app's console log."""
    r = subprocess.run(['az', 'containerapp', 'logs', 'show', '-n', APP_NAME, '-g', RG, '--type', 'console',
                        '--tail', str(tail), '--format', 'json'], capture_output=True, text=True)
    return checks.wrapper_events(r.stdout.splitlines(), kind)


response = clients['A'].responses.create(input=checks.prompt('business')).model_dump(warnings=False)
print(f"User A, business: {checks.classify(response, ROLES['A'], 'business')}, tools called: {checks.called_tools(response)}\n")
time.sleep(10)  # the log stream lags a few seconds

session = wrapper_events('initialize')[-1]
claims = wrapper_events('tools/call')[-1]
print(f"initialize : client={session['client']}  requested={session['requested']}  negotiated={session['negotiated']}")
print(f"MCP-Protocol-Version header on tools/call: {claims['mcp_protocol_version']}")
for key in ('idtyp', 'oid', 'azp', 'appid', 'roles', 'scp', 'aud', 'tool', 'decision'):
    if key in claims:
        print(f'  {key:<9} {claims[key]}')
print(f"\nazp is news-desk-agent's agent identity: {claims.get('azp') == agent_identity(ident.AGENT_NAME)['client_id']}")

User A, business: articles, tools called: [('get_business_headlines', {'max': 3, 'lang': 'en'})]

initialize : client=ToolServer  requested=2025-11-25  negotiated=2025-11-25
MCP-Protocol-Version header on tools/call: 2025-11-25
  idtyp     user
  oid       00000000-0000-0000-0000-000000000000
  azp       00000000-0000-0000-0000-000000000000
  roles     ['News.Read.All']
  scp       News.Read
  aud       00000000-0000-0000-0000-000000000000
  tool      get_business_headlines
  decision  allowed

azp is news-desk-agent's agent identity: True


Foundry's MCP client calls itself `ToolServer`, asks for protocol `2025-11-25` and sends `MCP-Protocol-Version: 2025-11-25` on each `tools/call`. For each response it opens a new session: `initialize`, a `GET /mcp` for a server-sent event stream (the stateless wrapper answers 405 and Foundry carries on), `tools/list`, then the calls. `azp` is the client id of `news-desk-agent`'s agent identity, not the blueprint's.

## Step 6: Run the test matrix as both users

Each case sends one question to the agent as one user and checks what comes back. Every question has the form "Give me three {category} headlines". The cases come from [`news_mcp_checks.py`](news_mcp_checks.py):

| # | User | Asks for | Expected outcome | Why |
|---|---|---|---|---|
| 1 | A | business headlines | `articles`: three business headlines | user A holds `News.Read.All`, so the wrapper lists all ten tools |
| 2 | B | business headlines | `hidden`: no tool call, and a reply that user B's access does not include business | the wrapper leaves `get_business_headlines` out of user B's `tools/list`, so the agent has no tool to call |
| 3 | B | sports headlines | `articles`: three sports headlines | user B holds `News.Read.Sports` |
| 4 | B | science headlines | `articles`: three science headlines | user B still holds `News.Read.Science`; Step 8 revokes it |

The outcomes the matrix can report:

| Outcome | Meaning |
|---|---|
| `articles` | a call for the requested category returned headlines |
| `hidden` | the server listed no tool for the category, and the agent called nothing (or called the tool anyway and Foundry refused it as not listed) |
| `denied` | a call came back with the denial text, which names the user's categories in GNews order |
| `substituted` | the agent fetched another category instead; the matrix fails |

No case on this track expects `denied`. Step 7 calls the hidden tool directly to show the denial text. On 08-05c-03, where everyone sees GNews's own tools, case 2 is `denied` instead of `hidden`.

The cell prints one row per case: the user, the category asked for, how many tools the agent listed, the tool it called, the outcome, the expected outcome, and PASS or FAIL. It stops with an error if a case did not get its expected outcome. Under the matrix it prints the wrapper's log for the same calls: one `tools/call` line per call with the decision, and one `upstream` line per call that reached GNews. Case 2 leaves no line at all, because the agent called nothing.

In [9]:
cases = (*checks.base_cases('wrapper'), checks.science_case('articles'))
records = checks.run_matrix(clients, cases, ROLES)
time.sleep(10)

events = [e for e in wrapper_events() if e['event'] in ('tools/call', 'upstream')]
calls = [i for i, e in enumerate(events) if e['event'] == 'tools/call']
made = sum(len(r['called']) for r in records)  # the hidden case makes none
who = {os.environ['NEWS_TEST_USER_ID']: 'B'}
print('\nWrapper log for the matrix:')
for e in events[calls[-made]:]:
    if e['event'] == 'tools/call':
        print(f"  tools/call  user {who.get(e['oid'], 'A')}  {e['tool']:<24} {e['decision']}")
    else:
        print(f"      upstream  {e['tool']}({', '.join(f'{k}={v}' for k, v in e['arguments'].items())})")

User  Asked for  Tools listed  Tool called             Outcome      Expected   Result  Tool result
----  ---------  ------------  ----------------------  -----------  ---------  ------  -----------
A     business             10  get_business_headlines  articles     articles   PASS    {"information":{"realTimeArticles":{"message":"On the Free plan, articles...
B     business              3  -                       hidden       hidden     PASS    I’m sorry, but my current news access doesn’t include business headlines.
B     sports                3  get_sports_headlines    articles     articles   PASS    {"information":{"realTimeArticles":{"message":"On the Free plan, articles...
B     science               3  get_science_headlines   articles     articles   PASS    {"information":{"realTimeArticles":{"message":"On the Free plan, articles...

Outcomes
  articles     a call for the requested category returned headlines
  denied       a call came back with the denial text, which names the u

**Why the instructions forbid another category.** With instructions that only said not to retry after a denial, user B's business request once came back with science headlines: the model had no business tool, so it used one it had. The instructions now add "If no news tool covers the category the user asks for, say that their access does not include it and do not fetch another category instead", and the matrix fails a `substituted` outcome so a regression shows.

## Your own questions

Ask the agent anything as either user: set `USER` and `QUESTION` and run the cell as often as you like. The cell prints the tools the agent saw, the calls it made and the reply. A reply that fetches articles uses one GNews request.

To try on this track:

- `What news tools do you have?` as A, then as B: ten tools, then three
- `Give me three business headlines` as B: no tool call, and a reply that user B's access does not include business
- `Give me three health headlines` as B: articles

In [10]:
USER = 'B'                                   # 'A' or 'B'
QUESTION = 'Give me three business headlines'

response = checks.ask(clients[USER], QUESTION)
print(f'Tools listed : {checks.listed_tools(response)}')
print(f'Tools called : {checks.called_tools(response)}')
print(f'Reply        : {checks.reply(response)}')

Tools listed : ['get_sports_headlines', 'get_science_headlines', 'get_health_headlines']
Tools called : []
Reply        : I don’t have access to a business-news tool in this setup, so I can’t fetch business headlines.


## Step 7: Compare each user's `tools/list`, through the agent and directly

The cell makes three checks:

1. **Through the agent.** The agent lists the server's tools before every response, so the matrix in Step 6 already recorded what each user saw. The cell prints those lists: ten tools for user A, three for user B.
2. **Directly.** Each user's own token for `api://news-mcp-{suffix}/News.Read` goes straight to the wrapper's `tools/list`, with no agent in between. The lists match the first check. The wrapper runs stateless, so the request needs no `initialize` first.
3. **A direct call to a hidden tool.** User B calls `get_business_headlines` with the same token. The wrapper answers with `isError: true` and the denial text, and its log shows the call denied with no `upstream` line after it.

The third check shows that the wrapper does not rely on hiding the tool: a client that skips `tools/list`, or holds an old list, still gets no business news.

In [11]:
for user in ('A', 'B'):
    listed = next(r['listed'] for r in records if r['user'] == user and r['listed'])
    print(f'User {user} through the agent: {len(listed)} tools  {listed}')
print()
tokens = {}
for user, cred in (('A', cred_a), ('B', cred_b)):
    tokens[user] = cred.get_token(f'{authz.app_id_uri(SUFFIX)}/{authz.SCOPE}').token
    tools = checks.list_tools_direct(WRAPPER_MCP_URL, tokens[user])
    print(f'User {user} directly       : {len(tools)} tools  {tools}')

hidden_tool = authz.tool_for('business')
result = checks.call_tool_direct(WRAPPER_MCP_URL, tokens['B'], hidden_tool, {'max': 3})
denial = result['content'][0]['text']
print(f"\nUser B calls {hidden_tool} directly: isError={result['isError']}\n  {denial}")
assert result['isError'] and denial == authz.denial_message(ROLES['B'])
time.sleep(10)  # the log stream lags a few seconds
events = [e for e in wrapper_events() if e['event'] in ('tools/call', 'upstream')]
last = max(i for i, e in enumerate(events) if e['event'] == 'tools/call')
print(f"Wrapper log: tools/call {events[last]['tool']} {events[last]['decision']}, "
      f"upstream lines after it: {sum(e['event'] == 'upstream' for e in events[last + 1:])}")

User A through the agent: 10 tools  ['get_general_headlines', 'get_world_headlines', 'get_nation_headlines', 'get_business_headlines', 'get_technology_headlines', 'get_entertainment_headlines', 'get_sports_headlines', 'get_science_headlines', 'get_health_headlines', 'search_news']
User B through the agent: 3 tools  ['get_sports_headlines', 'get_science_headlines', 'get_health_headlines']

User A directly       : 10 tools  ['get_general_headlines', 'get_world_headlines', 'get_nation_headlines', 'get_business_headlines', 'get_technology_headlines', 'get_entertainment_headlines', 'get_sports_headlines', 'get_science_headlines', 'get_health_headlines', 'search_news']
User B directly       : 3 tools  ['get_sports_headlines', 'get_science_headlines', 'get_health_headlines']

User B calls get_business_headlines directly: isError=True
  Your access covers: sports, science, health. Do not retry other categories.
Wrapper log: tools/call get_business_headlines denied, upstream lines after it: 0


## Step 8: Revoke user B's `News.Read.Science`

The cell deletes user B's `News.Read.Science` assignment on `news-mcp-{suffix}`, so that 08-05c-03 can show what a revocation does. It prints the time of the revocation and the roles user B still holds: health and sports.

- **What changes:** Entra leaves `News.Read.Science` out of the next token it issues for user B.
- **What does not change at once:** a token Foundry obtained and cached before the revocation still carries the role until it expires.
- **What 08-05c-03 expects:** it reads user B's roles from Graph at sign-in, so its science case expects a denial. Foundry has not yet obtained a token for user B through the `news-mcp-apim` connection, so the first one it gets carries the new roles.

To give the role back for another run of both tracks, run [08-05c-04](08-05c-04-reset-for-another-run.ipynb) after 08-05c-03.

In [ ]:
assigned_to = f'/servicePrincipals/{RESOURCE_SP_ID}/appRoleAssignedTo'
science = authz.permission_id(SUFFIX, 'News.Read.Science')
for a in graph('GET', assigned_to)['value']:
    if a['principalId'] == TEST_USER_ID and a['appRoleId'] == science:
        graph('DELETE', f"{assigned_to}/{a['id']}")
revoked_at = datetime.now(timezone.utc).isoformat(timespec='seconds')
roles_b = sorted(r for r, i in ((r, authz.permission_id(SUFFIX, r)) for r in authz.ROLES)
                 if i in {a['appRoleId'] for a in graph('GET', assigned_to)['value'] if a['principalId'] == TEST_USER_ID})
print(f'Revoked News.Read.Science from user B at {revoked_at}')
print(f'User B now holds: {roles_b}')

## Cleanup

Leave everything in place for 08-05c-03. The full teardown is the Cleanup section of [08-05c-01](08-05c-01-identity-and-agent.ipynb), which removes this notebook's resources too. To remove only Track A, uncomment the cell.

In [ ]:
# print('news-mcp-wrapper', call('DELETE', ident.connection_url(SUBSCRIPTION_ID, SUFFIX, 'news-mcp-wrapper', CONNECTION_API), ARM).status_code)
# subprocess.run(['az', 'group', 'delete', '-n', RG, '--yes'], check=True)